In [1]:
# ============================================================
# Validation Step 1 (Automated Rule-Based Validation) — FULLY ADJUSTED
#
# Fixes included:
#   - V11 now uses the same Layer 2 applicability mask as V8-V14
#   - V11 checks:
#       pre_invocation + invocation_execution_window + post_invocation
#       == study_run_duration_seconds
#   - V31 now checks count–list consistency instead of raw presence:
#       blank list is acceptable when candidate count == 0
#   - V52 now validates the latest Stage 3 invocation priority-source labels
#       and checks contextual coherence against candidate counts / selected invocation
#
# Reads from:
#   C:\Android Mobile App\ICST2026_Ext\MainDataset.csv
#   C:\Android Mobile App\ICST2026_Ext\run_steps_v16_stage3_breakdown.csv
#     or C:\Android Mobile App\ICST2026_Ext\run_steps_v16_stage3_breakdown.zip
#
# Writes to:
#   C:\Android Mobile App\ICST2026_Ext\Validation_Step_1\
#
# Output files:
#   validation_step1_summary.csv
#   validation_step1_style_summary.csv
#   validation_step1_controlled_subset_summary.csv
#   validation_step1_style_controlled_subset_summary.csv
#   validation_step1_record_flags.csv
#   validation_step1_cutpoint_mismatches.csv
#   validation_step1_step_match_issues.csv
#   validation_step1_auxiliary_issues.csv
#   validation_step1_all_mismatches_controlled_subset_only.csv
# ============================================================

from pathlib import Path
import zipfile
import numpy as np
import pandas as pd

# ----------------------------
# Paths
# ----------------------------
BASE_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")
IN_MAIN = BASE_DIR / "MainDataset.csv"
IN_STEPS_CSV = BASE_DIR / "run_steps_v16_stage3_breakdown.csv"
IN_STEPS_ZIP = BASE_DIR / "run_steps_v16_stage3_breakdown.zip"

OUT_DIR = BASE_DIR / "Validation_Step_1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Config
# ----------------------------
EPS = 1e-6

# ----------------------------
# Helpers
# ----------------------------
def to_dt(s):
    return pd.to_datetime(s, utc=True, errors="coerce")

def norm_bool_series(s):
    if s is None:
        return pd.Series(dtype="boolean")
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)
    s = s.astype(str).str.strip().str.lower()
    return s.isin(["true", "1", "yes", "y", "t"])

def yes_no_from_mask(mask):
    return np.where(mask, "Yes", "No")

def make_flag(applicable_mask, ok_mask):
    """
    Return 'ok' / 'mismatch' / 'missing'
    """
    out = np.where(applicable_mask, np.where(ok_mask, "ok", "mismatch"), "missing")
    return pd.Series(out, index=applicable_mask.index if hasattr(applicable_mask, "index") else None)

def summarize_flags(df_in, flag_cols, group_cols=None):
    rows = []
    if group_cols is None:
        for c in flag_cols:
            vc = df_in[c].value_counts(dropna=False)
            total = len(df_in)
            ok = int(vc.get("ok", 0))
            mismatch = int(vc.get("mismatch", 0))
            missing = int(vc.get("missing", 0))
            rows.append({
                "check_name": c,
                "rows_total": total,
                "ok_count": ok,
                "mismatch_count": mismatch,
                "missing_count": missing,
                "ok_pct": round(ok * 100.0 / total, 4) if total else 0.0,
                "mismatch_pct": round(mismatch * 100.0 / total, 4) if total else 0.0,
                "missing_pct": round(missing * 100.0 / total, 4) if total else 0.0,
            })
        return pd.DataFrame(rows)

    grouped = df_in.groupby(group_cols, dropna=False, sort=True)
    for key, sub in grouped:
        if not isinstance(key, tuple):
            key = (key,)
        key_dict = dict(zip(group_cols, key))
        total = len(sub)
        for c in flag_cols:
            vc = sub[c].value_counts(dropna=False)
            ok = int(vc.get("ok", 0))
            mismatch = int(vc.get("mismatch", 0))
            missing = int(vc.get("missing", 0))
            row = {
                **key_dict,
                "check_name": c,
                "rows_total": total,
                "ok_count": ok,
                "mismatch_count": mismatch,
                "missing_count": missing,
                "ok_pct": round(ok * 100.0 / total, 4) if total else 0.0,
                "mismatch_pct": round(mismatch * 100.0 / total, 4) if total else 0.0,
                "missing_pct": round(missing * 100.0 / total, 4) if total else 0.0,
            }
            rows.append(row)
    return pd.DataFrame(rows)

def first_existing(df, cols, default=np.nan):
    for c in cols:
        if c in df.columns:
            return df[c]
    return pd.Series([default] * len(df), index=df.index)

def has_nonempty(x):
    if pd.isna(x):
        return False
    s = str(x).strip()
    return s != "" and s.lower() != "nan"

def present_flag(series):
    return series.map(has_nonempty)

def normalize_listish_text(x):
    """
    Normalize list-like text fields.
    Treat NaN/blank/[] as empty.
    """
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return ""
    if s in {"[]", "[ ]"}:
        return ""
    return s

def canonicalize_priority_source(x):
    """
    Canonicalize selected invocation priority-source labels to the current Stage 3 scheme.
    Supports both current labels and older aliases.
    """
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    if s == "" or s == "nan":
        return ""

    mapping = {
        # current/latest labels
        "stage1_anchor_match": "stage1_anchor_match",
        "explicit_instru_execution_start": "explicit_instru_execution_start",
        "custom_followed_file_instru": "custom_followed_file_instru",
        "custom_stage1_supported_exec": "custom_stage1_supported_exec",
        "missing": "missing",

        # older / alias labels
        "anchor_match": "stage1_anchor_match",
        "stage1_anchor": "stage1_anchor_match",
        "stage1_anchor_reasoned": "stage1_anchor_match",

        "explicit_instru": "explicit_instru_execution_start",
        "explicit_instrumentation": "explicit_instru_execution_start",
        "explicit_instru_start": "explicit_instru_execution_start",
        "explicit_execution_start": "explicit_instru_execution_start",

        "custom_supported": "custom_stage1_supported_exec",
    }
    return mapping.get(s, s)

# ----------------------------
# Read inputs
# ----------------------------
df = pd.read_csv(IN_MAIN)

if IN_STEPS_CSV.exists():
    steps = pd.read_csv(IN_STEPS_CSV)
elif IN_STEPS_ZIP.exists():
    with zipfile.ZipFile(IN_STEPS_ZIP) as zf:
        names = zf.namelist()
        if not names:
            raise FileNotFoundError("run_steps_v16_stage3_breakdown.zip is empty.")
        with zf.open(names[0]) as f:
            steps = pd.read_csv(f)
else:
    raise FileNotFoundError("Could not find run_steps_v16_stage3_breakdown.csv or .zip")

# ----------------------------
# Normalize a few common columns
# ----------------------------
if "style" not in df.columns and "target_style" in df.columns:
    df["style"] = df["target_style"]

if "run_attempt" in df.columns and "attempt" not in df.columns:
    df["attempt"] = df["run_attempt"]

if "controller_attempt_eq_1" in df.columns:
    controller_attempt_eq_1 = norm_bool_series(df["controller_attempt_eq_1"])
else:
    controller_attempt_eq_1 = pd.to_numeric(first_existing(df, ["attempt", "run_attempt"]), errors="coerce").eq(1)

if "controller_run_verdict_complete" in df.columns:
    controller_run_verdict_complete = norm_bool_series(df["controller_run_verdict_complete"])
else:
    # Fallback:
    # Treat success/failure as verdict-complete, everything else as not-complete.
    rc = first_existing(df, ["run_conclusion", "conclusion"]).astype(str).str.lower()
    controller_run_verdict_complete = rc.isin(["success", "failure", "failed"])

df["controlled_subset"] = yes_no_from_mask(controller_attempt_eq_1 & controller_run_verdict_complete)

# ----------------------------
# Parse datetime fields
# ----------------------------
run_start = to_dt(first_existing(df, ["study_run_boundary_start_at"]))
run_end = to_dt(first_existing(df, ["study_run_boundary_end_at"]))

inv_start = to_dt(first_existing(df, ["study_matched_invocation_step_started_at"]))
inv_end = to_dt(first_existing(df, ["study_matched_invocation_step_completed_at"]))
exec_end_start = to_dt(first_existing(df, ["study_invocation_execution_end_step_started_at"]))
exec_end_end = to_dt(first_existing(df, ["study_invocation_execution_end_step_completed_at"]))

window_start = to_dt(first_existing(df, ["study_invocation_execution_window_started_at"]))
window_end = to_dt(first_existing(df, ["study_invocation_execution_window_ended_at"]))

# ----------------------------
# Numeric fields
# ----------------------------
run_dur = pd.to_numeric(first_existing(df, ["study_run_duration_seconds"]), errors="coerce")

layer1_pre = pd.to_numeric(first_existing(df, ["study_layer1_time_to_instrumentation_envelope_seconds"]), errors="coerce")
layer1_mid = pd.to_numeric(first_existing(df, ["study_layer1_instrumentation_job_envelope_seconds"]), errors="coerce")
layer1_post = pd.to_numeric(first_existing(df, ["study_layer1_post_instrumentation_tail_seconds"]), errors="coerce")

l2_pre = pd.to_numeric(first_existing(df, ["study_pre_invocation_selected_stage3_seconds"]), errors="coerce")
l2_win = pd.to_numeric(first_existing(df, ["study_invocation_execution_window_selected_stage3_seconds"]), errors="coerce")
l2_post = pd.to_numeric(first_existing(df, ["study_post_invocation_selected_stage3_seconds"]), errors="coerce")

# ----------------------------
# Step-level source prep for V15-V18
# ----------------------------
steps["selected_invocation_cutpoint_bool"] = norm_bool_series(first_existing(steps, ["selected_invocation_cutpoint"]))
steps["selected_execution_end_cutpoint_bool"] = norm_bool_series(first_existing(steps, ["selected_execution_end_cutpoint"]))

for col in [
    "run_id", "job_ordinal_in_run", "step_ordinal_in_job", "step_name", "job_name", "started_at", "completed_at"
]:
    if col not in steps.columns:
        steps[col] = np.nan

steps["started_at_dt"] = to_dt(steps["started_at"])
steps["completed_at_dt"] = to_dt(steps["completed_at"])

inv_sel = (
    steps.loc[steps["selected_invocation_cutpoint_bool"]]
    .groupby("run_id", dropna=False)
    .agg(
        inv_selected_count=("selected_invocation_cutpoint_bool", "sum"),
        inv_step_name_source=("step_name", "first"),
        inv_job_name_source=("job_name", "first"),
        inv_job_ordinal_source=("job_ordinal_in_run", "first"),
        inv_step_ordinal_source=("step_ordinal_in_job", "first"),
        inv_started_at_source=("started_at_dt", "first"),
        inv_completed_at_source=("completed_at_dt", "first"),
    )
    .reset_index()
)

exe_sel = (
    steps.loc[steps["selected_execution_end_cutpoint_bool"]]
    .groupby("run_id", dropna=False)
    .agg(
        exe_selected_count=("selected_execution_end_cutpoint_bool", "sum"),
        exe_step_name_source=("step_name", "first"),
        exe_job_name_source=("job_name", "first"),
        exe_job_ordinal_source=("job_ordinal_in_run", "first"),
        exe_step_ordinal_source=("step_ordinal_in_job", "first"),
        exe_started_at_source=("started_at_dt", "first"),
        exe_completed_at_source=("completed_at_dt", "first"),
    )
    .reset_index()
)

df = df.merge(inv_sel, on="run_id", how="left")
df = df.merge(exe_sel, on="run_id", how="left")

# ----------------------------
# Convenience series for stored source-matched fields
# ----------------------------
stored_inv_step_name = first_existing(df, ["study_matched_invocation_step_name"])
stored_inv_job_name = first_existing(df, ["study_matched_invocation_job_name"])
stored_inv_job_ord = pd.to_numeric(first_existing(df, ["study_matched_invocation_job_ordinal_in_run"]), errors="coerce")
stored_inv_step_ord = pd.to_numeric(first_existing(df, ["study_matched_invocation_step_ordinal_in_job"]), errors="coerce")
stored_inv_started = to_dt(first_existing(df, ["study_matched_invocation_step_started_at"]))
stored_inv_completed = to_dt(first_existing(df, ["study_matched_invocation_step_completed_at"]))

stored_exe_step_name = first_existing(df, ["study_invocation_execution_end_step_name"])
stored_exe_job_name = first_existing(df, ["study_invocation_execution_end_job_name"])
stored_exe_job_ord = pd.to_numeric(first_existing(df, ["study_invocation_execution_end_job_ordinal_in_run"]), errors="coerce")
stored_exe_step_ord = pd.to_numeric(first_existing(df, ["study_invocation_execution_end_step_ordinal_in_job"]), errors="coerce")
stored_exe_started = to_dt(first_existing(df, ["study_invocation_execution_end_step_started_at"]))
stored_exe_completed = to_dt(first_existing(df, ["study_invocation_execution_end_step_completed_at"]))

# ----------------------------
# Rule flags
# ----------------------------

# V1 Key uniqueness
key_cols = [c for c in ["full_name", "run_id", "style"] if c in df.columns]
if len(key_cols) < 3:
    raise ValueError(f"Expected key columns full_name, run_id, style. Found: {key_cols}")
dup_mask = df.duplicated(subset=key_cols, keep=False)
df["v1_key_uniqueness_flag"] = np.where(dup_mask, "mismatch", "ok")

# V2 Required identifiers present
required_id_cols = [c for c in ["full_name", "run_id", "workflow_id", "workflow_identifier", "workflow_path"] if c in df.columns]
required_id_present = pd.concat([present_flag(df[c]) for c in required_id_cols], axis=1).all(axis=1)
df["v2_required_ids_flag"] = np.where(required_id_present, "ok", "mismatch")

# V3 Run bounds ordered
v3_app = run_start.notna() & run_end.notna()
v3_ok = run_start <= run_end
df["v3_run_bounds_order_flag"] = make_flag(v3_app, v3_ok)

# V4 Window bounds ordered
v4_app = window_start.notna() & window_end.notna()
v4_ok = window_start <= window_end
df["v4_window_bounds_order_flag"] = make_flag(v4_app, v4_ok)

# V5 Cutpoint temporal order
v5_app = run_start.notna() & inv_start.notna() & exec_end_end.notna() & run_end.notna()
v5_ok = (run_start <= inv_start) & (inv_start <= exec_end_end) & (exec_end_end <= run_end)
df["v5_cutpoint_temporal_order_flag"] = make_flag(v5_app, v5_ok)

# V6 Window inside run
v6_app = run_start.notna() & window_start.notna() & window_end.notna() & run_end.notna()
v6_ok = (run_start <= window_start) & (window_start <= window_end) & (window_end <= run_end)
df["v6_window_inside_run_flag"] = make_flag(v6_app, v6_ok)

# V7 Run duration non-negative
v7_app = run_dur.notna()
v7_ok = run_dur >= -EPS
df["v7_run_duration_nonnegative_flag"] = make_flag(v7_app, v7_ok)

# Shared Layer 2 applicability mask for V8-V14 INCLUDING FIXED V11
layer2_app = l2_pre.notna() & l2_win.notna() & l2_post.notna() & run_dur.notna()

# V8 Layer 2 non-negative
v8_ok = (l2_pre >= -EPS) & (l2_win >= -EPS) & (l2_post >= -EPS)
df["v8_layer2_nonnegative_flag"] = make_flag(layer2_app, v8_ok)

# V9 Layer 2 components bounded by run duration
v9_ok = (l2_pre <= run_dur + EPS) & (l2_win <= run_dur + EPS) & (l2_post <= run_dur + EPS)
df["v9_layer2_bounded_by_run_flag"] = make_flag(layer2_app, v9_ok)

# V10 Layer 1 sum-to-run equation
v10_app = layer1_pre.notna() & layer1_mid.notna() & layer1_post.notna() & run_dur.notna()
v10_ok = ((layer1_pre + layer1_mid + layer1_post) - run_dur).abs() <= EPS
df["v10_layer1_sum_to_run_flag"] = make_flag(v10_app, v10_ok)

# V11 FIXED: Layer 2 sum-to-run equation
v11_ok = ((l2_pre + l2_win + l2_post) - run_dur).abs() <= EPS
df["v11_window_decomposition_flag"] = make_flag(layer2_app, v11_ok)

# V12 Pre-invocation recomputation
v12_app = run_start.notna() & inv_start.notna() & l2_pre.notna()
v12_recomp = (inv_start - run_start).dt.total_seconds()
v12_ok = (v12_recomp - l2_pre).abs() <= EPS
df["v12_pre_invocation_recompute_flag"] = make_flag(v12_app, v12_ok)

# V13 Invocation-window recomputation
v13_app = inv_start.notna() & exec_end_end.notna() & l2_win.notna()
v13_recomp = (exec_end_end - inv_start).dt.total_seconds()
v13_ok = (v13_recomp - l2_win).abs() <= EPS
df["v13_invocation_window_recompute_flag"] = make_flag(v13_app, v13_ok)

# V14 Post-invocation recomputation
v14_app = exec_end_end.notna() & run_end.notna() & l2_post.notna()
v14_recomp = (run_end - exec_end_end).dt.total_seconds()
v14_ok = (v14_recomp - l2_post).abs() <= EPS
df["v14_post_invocation_recompute_flag"] = make_flag(v14_app, v14_ok)

# V15 Exactly one invocation cutpoint selected
v15_app = df["inv_selected_count"].notna()
v15_ok = df["inv_selected_count"].fillna(0).eq(1)
df["v15_unique_invocation_cutpoint_flag"] = make_flag(v15_app, v15_ok)

# V16 Exactly one execution-end cutpoint selected
v16_app = df["exe_selected_count"].notna()
v16_ok = df["exe_selected_count"].fillna(0).eq(1)
df["v16_unique_execution_end_cutpoint_flag"] = make_flag(v16_app, v16_ok)

# V17 Invocation-step source match
v17_app = (
    df["inv_selected_count"].fillna(0).eq(1)
    & stored_inv_step_name.map(has_nonempty)
    & stored_inv_job_name.map(has_nonempty)
    & stored_inv_started.notna()
)
v17_ok = (
    stored_inv_step_name.astype(str).fillna("") == df["inv_step_name_source"].astype(str).fillna("")
) & (
    stored_inv_job_name.astype(str).fillna("") == df["inv_job_name_source"].astype(str).fillna("")
) & (
    stored_inv_job_ord.fillna(-999999) == pd.to_numeric(df["inv_job_ordinal_source"], errors="coerce").fillna(-999999)
) & (
    stored_inv_step_ord.fillna(-999999) == pd.to_numeric(df["inv_step_ordinal_source"], errors="coerce").fillna(-999999)
) & (
    stored_inv_started == to_dt(df["inv_started_at_source"])
) & (
    stored_inv_completed == to_dt(df["inv_completed_at_source"])
)
df["v17_invocation_step_source_match_flag"] = make_flag(v17_app, v17_ok)

# V18 Execution-end-step source match
v18_app = (
    df["exe_selected_count"].fillna(0).eq(1)
    & stored_exe_step_name.map(has_nonempty)
    & stored_exe_job_name.map(has_nonempty)
    & stored_exe_started.notna()
)
v18_ok = (
    stored_exe_step_name.astype(str).fillna("") == df["exe_step_name_source"].astype(str).fillna("")
) & (
    stored_exe_job_name.astype(str).fillna("") == df["exe_job_name_source"].astype(str).fillna("")
) & (
    stored_exe_job_ord.fillna(-999999) == pd.to_numeric(df["exe_job_ordinal_source"], errors="coerce").fillna(-999999)
) & (
    stored_exe_step_ord.fillna(-999999) == pd.to_numeric(df["exe_step_ordinal_source"], errors="coerce").fillna(-999999)
) & (
    stored_exe_started == to_dt(df["exe_started_at_source"])
) & (
    stored_exe_completed == to_dt(df["exe_completed_at_source"])
)
df["v18_execution_end_step_source_match_flag"] = make_flag(v18_app, v18_ok)

# V19 Style scope validity
allowed_styles = {"Community", "GMD", "Third-Party", "Custom"}
v19_app = df["style"].notna()
v19_ok = df["style"].astype(str).isin(allowed_styles)
df["v19_style_scope_valid_flag"] = make_flag(v19_app, v19_ok)

# Presence-style utility
def presence_rule(series):
    app = pd.Series(True, index=df.index)
    ok = present_flag(series)
    return make_flag(app, ok)

# V20-V21 base/robust presence
df["v20_base_flag_present"] = presence_rule(first_existing(df, ["Base"]))
df["v21_robust_flag_present"] = presence_rule(first_existing(df, ["Robust"]))

# V22-V30 style-level presence fields
df["v22_style_distinct_job_count_present"] = presence_rule(first_existing(df, ["study_style_distinct_job_count"]))
df["v23_style_distinct_job_base_count_present"] = presence_rule(first_existing(df, ["study_style_distinct_job_base_name_count"]))
df["v24_style_matrix_like_job_count_present"] = presence_rule(first_existing(df, ["study_style_matrix_like_job_count"]))
df["v25_style_matrix_expansion_flag_present"] = presence_rule(first_existing(df, ["study_style_matrix_expanded_flag"]))
df["v26_style_parallel_same_style_flag_present"] = presence_rule(first_existing(df, ["study_style_parallel_same_style_flag"]))
df["v27_style_max_parallel_jobs_present"] = presence_rule(first_existing(df, ["study_style_max_parallel_jobs"]))
df["v28_style_repeated_same_style_flag_present"] = presence_rule(first_existing(df, ["study_style_repeated_same_style_flag"]))
df["v29_style_invocation_candidate_count_present"] = presence_rule(first_existing(df, ["study_style_invocation_candidate_step_count_proxy"]))
df["v30_style_distinct_invocation_step_name_count_present"] = presence_rule(first_existing(df, ["study_style_distinct_invocation_step_name_count_proxy"]))

# V31 FIXED: style-level invocation-candidate step-name list consistency
# Blank list is acceptable when candidate count == 0.
# A nonblank list is required when candidate count > 0.
v31_count = pd.to_numeric(
    first_existing(df, ["study_style_invocation_candidate_step_count_proxy"]),
    errors="coerce"
)
v31_list_raw = first_existing(df, ["study_style_invocation_candidate_step_names_proxy"])
v31_list_norm = v31_list_raw.map(normalize_listish_text)
v31_list_present = v31_list_norm.ne("")

v31_app = v31_count.notna()
v31_ok = (
    ((v31_count == 0) & (~v31_list_present)) |
    ((v31_count > 0) & v31_list_present)
)
df["v31_style_invocation_step_name_list_present"] = make_flag(v31_app, v31_ok)

# V32 style complexity class present
df["v32_style_complexity_class_present"] = presence_rule(first_existing(df, ["study_style_same_style_complexity_class"]))

# V33-V42 auxiliary presence fields
df["v33_invocation_candidate_total_count_present"] = presence_rule(first_existing(df, ["study_invocation_candidate_count_total"]))
df["v34_stage1_anchor_candidate_count_present"] = presence_rule(first_existing(df, ["study_stage1_anchor_candidate_count"]))
df["v35_explicit_instru_candidate_count_present"] = presence_rule(first_existing(df, ["study_explicit_instru_candidate_count"]))
df["v36_custom_supported_candidate_count_present"] = presence_rule(first_existing(df, ["study_custom_supported_candidate_count"]))
df["v37_distinct_invocation_candidate_step_name_count_present"] = presence_rule(first_existing(df, ["study_distinct_invocation_candidate_step_name_count"]))
df["v38_distinct_invocation_candidate_job_count_present"] = presence_rule(first_existing(df, ["study_distinct_invocation_candidate_job_count"]))
df["v39_selected_invocation_priority_source_present"] = presence_rule(first_existing(df, ["study_selected_invocation_priority_source"]))
df["v40_execution_window_candidate_count_present"] = presence_rule(first_existing(df, ["study_execution_window_candidate_count"]))
df["v41_execution_window_distinct_job_count_present"] = presence_rule(first_existing(df, ["study_execution_window_distinct_job_count"]))
df["v42_cross_job_execution_window_flag_present"] = presence_rule(first_existing(df, ["study_cross_job_execution_window_flag"]))

# Numeric boundedness / consistency rules
style_distinct_job_count = pd.to_numeric(first_existing(df, ["study_style_distinct_job_count"]), errors="coerce")
style_distinct_job_base_count = pd.to_numeric(first_existing(df, ["study_style_distinct_job_base_name_count"]), errors="coerce")
style_matrix_like_count = pd.to_numeric(first_existing(df, ["study_style_matrix_like_job_count"]), errors="coerce")
style_max_parallel_jobs = pd.to_numeric(first_existing(df, ["study_style_max_parallel_jobs"]), errors="coerce")
style_inv_cand_proxy = pd.to_numeric(first_existing(df, ["study_style_invocation_candidate_step_count_proxy"]), errors="coerce")
style_inv_distinct_name_proxy = pd.to_numeric(first_existing(df, ["study_style_distinct_invocation_step_name_count_proxy"]), errors="coerce")

inv_total = pd.to_numeric(first_existing(df, ["study_invocation_candidate_count_total"]), errors="coerce")
inv_anchor_count = pd.to_numeric(first_existing(df, ["study_stage1_anchor_candidate_count"]), errors="coerce")
inv_explicit_count = pd.to_numeric(first_existing(df, ["study_explicit_instru_candidate_count"]), errors="coerce")
inv_custom_count = pd.to_numeric(first_existing(df, ["study_custom_supported_candidate_count"]), errors="coerce")
inv_distinct_step_name_count = pd.to_numeric(first_existing(df, ["study_distinct_invocation_candidate_step_name_count"]), errors="coerce")
exec_window_candidate_count = pd.to_numeric(first_existing(df, ["study_execution_window_candidate_count"]), errors="coerce")
exec_window_distinct_job_count = pd.to_numeric(first_existing(df, ["study_execution_window_distinct_job_count"]), errors="coerce")

# V43
v43_app = style_distinct_job_base_count.notna() & style_distinct_job_count.notna()
v43_ok = style_distinct_job_base_count <= style_distinct_job_count
df["v43_job_base_count_bounded_flag"] = make_flag(v43_app, v43_ok)

# V44
v44_app = style_matrix_like_count.notna() & style_distinct_job_count.notna()
v44_ok = style_matrix_like_count <= style_distinct_job_count
df["v44_matrix_like_count_bounded_flag"] = make_flag(v44_app, v44_ok)

# V45
v45_app = style_max_parallel_jobs.notna() & style_distinct_job_count.notna()
v45_ok = style_max_parallel_jobs <= style_distinct_job_count
df["v45_max_parallel_jobs_bounded_flag"] = make_flag(v45_app, v45_ok)

# V46
v46_app = style_inv_distinct_name_proxy.notna() & style_inv_cand_proxy.notna()
v46_ok = style_inv_distinct_name_proxy <= style_inv_cand_proxy
df["v46_distinct_invocation_name_proxy_bounded_flag"] = make_flag(v46_app, v46_ok)

# V47
v47_app = inv_total.notna() & inv_anchor_count.notna() & inv_explicit_count.notna() & inv_custom_count.notna()
v47_ok = (inv_anchor_count + inv_explicit_count + inv_custom_count) <= inv_total
df["v47_invocation_candidate_partition_flag"] = make_flag(v47_app, v47_ok)

# V48
v48_app = exec_window_distinct_job_count.notna() & exec_window_candidate_count.notna()
v48_ok = exec_window_distinct_job_count <= exec_window_candidate_count
df["v48_execution_window_distinct_job_bounded_flag"] = make_flag(v48_app, v48_ok)

# V49
style_matrix_expanded = norm_bool_series(first_existing(df, ["study_style_matrix_expanded_flag"]))
style_repeated_same_style = norm_bool_series(first_existing(df, ["study_style_repeated_same_style_flag"]))
v49_app = present_flag(first_existing(df, ["study_style_matrix_expanded_flag"])) & present_flag(first_existing(df, ["study_style_repeated_same_style_flag"]))
v49_ok = (~style_matrix_expanded) | style_repeated_same_style
df["v49_matrix_expanded_implies_repeated_flag"] = make_flag(v49_app, v49_ok)

# V50
cross_job_exec_window = norm_bool_series(first_existing(df, ["study_cross_job_execution_window_flag"]))
v50_app = present_flag(first_existing(df, ["study_cross_job_execution_window_flag"])) & exec_window_distinct_job_count.notna()
v50_ok = (~cross_job_exec_window & exec_window_distinct_job_count.fillna(0).le(1)) | (
    cross_job_exec_window & exec_window_distinct_job_count.fillna(0).ge(2)
)
df["v50_cross_job_window_flag_consistency"] = make_flag(v50_app, v50_ok)

# V51
style_parallel_same_style = norm_bool_series(first_existing(df, ["study_style_parallel_same_style_flag"]))
v51_app = present_flag(first_existing(df, ["study_style_parallel_same_style_flag"])) & style_max_parallel_jobs.notna()
v51_ok = (~style_parallel_same_style & style_max_parallel_jobs.fillna(0).le(1)) | (
    style_parallel_same_style & style_max_parallel_jobs.fillna(0).ge(2)
)
df["v51_parallel_same_style_flag_consistency"] = make_flag(v51_app, v51_ok)

# V52 FIXED/UPGRADED: selected invocation priority source validity + contextual coherence
priority_source_raw = first_existing(df, ["study_selected_invocation_priority_source"])
priority_source_canon = priority_source_raw.map(canonicalize_priority_source)

stage1_anchor_candidate_count = pd.to_numeric(
    first_existing(df, ["study_stage1_anchor_candidate_count"]),
    errors="coerce"
)
explicit_instru_candidate_count = pd.to_numeric(
    first_existing(df, ["study_explicit_instru_candidate_count"]),
    errors="coerce"
)
custom_supported_candidate_count = pd.to_numeric(
    first_existing(df, ["study_custom_supported_candidate_count"]),
    errors="coerce"
)
invocation_candidate_total_count = pd.to_numeric(
    first_existing(df, ["study_invocation_candidate_count_total"]),
    errors="coerce"
)

matched_invocation_started_at = to_dt(first_existing(df, ["study_matched_invocation_step_started_at"]))

valid_priority_sources = {
    "stage1_anchor_match",
    "explicit_instru_execution_start",
    "custom_followed_file_instru",
    "custom_stage1_supported_exec",
    "missing",
}

v52_app = priority_source_canon.map(has_nonempty)
v52_label_valid = priority_source_canon.isin(valid_priority_sources)

v52_context_ok = (
    ((priority_source_canon == "missing") & matched_invocation_started_at.isna()) |

    ((priority_source_canon == "stage1_anchor_match")
        & matched_invocation_started_at.notna()
        & stage1_anchor_candidate_count.fillna(0).ge(1)
        & invocation_candidate_total_count.fillna(0).ge(1)) |

    ((priority_source_canon == "explicit_instru_execution_start")
        & matched_invocation_started_at.notna()
        & explicit_instru_candidate_count.fillna(0).ge(1)
        & invocation_candidate_total_count.fillna(0).ge(1)) |

    ((priority_source_canon == "custom_followed_file_instru")
        & matched_invocation_started_at.notna()
        & invocation_candidate_total_count.fillna(0).ge(1)) |

    ((priority_source_canon == "custom_stage1_supported_exec")
        & matched_invocation_started_at.notna()
        & custom_supported_candidate_count.fillna(0).ge(1)
        & invocation_candidate_total_count.fillna(0).ge(1))
)

v52_ok = v52_label_valid & v52_context_ok
df["v52_selected_invocation_priority_source_validity"] = make_flag(v52_app, v52_ok)

# V53 style-level complexity class presence
df["v53_style_level_complexity_class_presence"] = presence_rule(first_existing(df, ["study_style_same_style_complexity_class"]))

# ----------------------------
# Flag list in rule order
# ----------------------------
flag_cols = [
    "v1_key_uniqueness_flag",
    "v2_required_ids_flag",
    "v3_run_bounds_order_flag",
    "v4_window_bounds_order_flag",
    "v5_cutpoint_temporal_order_flag",
    "v6_window_inside_run_flag",
    "v7_run_duration_nonnegative_flag",
    "v8_layer2_nonnegative_flag",
    "v9_layer2_bounded_by_run_flag",
    "v10_layer1_sum_to_run_flag",
    "v11_window_decomposition_flag",
    "v12_pre_invocation_recompute_flag",
    "v13_invocation_window_recompute_flag",
    "v14_post_invocation_recompute_flag",
    "v15_unique_invocation_cutpoint_flag",
    "v16_unique_execution_end_cutpoint_flag",
    "v17_invocation_step_source_match_flag",
    "v18_execution_end_step_source_match_flag",
    "v19_style_scope_valid_flag",
    "v20_base_flag_present",
    "v21_robust_flag_present",
    "v22_style_distinct_job_count_present",
    "v23_style_distinct_job_base_count_present",
    "v24_style_matrix_like_job_count_present",
    "v25_style_matrix_expansion_flag_present",
    "v26_style_parallel_same_style_flag_present",
    "v27_style_max_parallel_jobs_present",
    "v28_style_repeated_same_style_flag_present",
    "v29_style_invocation_candidate_count_present",
    "v30_style_distinct_invocation_step_name_count_present",
    "v31_style_invocation_step_name_list_present",
    "v32_style_complexity_class_present",
    "v33_invocation_candidate_total_count_present",
    "v34_stage1_anchor_candidate_count_present",
    "v35_explicit_instru_candidate_count_present",
    "v36_custom_supported_candidate_count_present",
    "v37_distinct_invocation_candidate_step_name_count_present",
    "v38_distinct_invocation_candidate_job_count_present",
    "v39_selected_invocation_priority_source_present",
    "v40_execution_window_candidate_count_present",
    "v41_execution_window_distinct_job_count_present",
    "v42_cross_job_execution_window_flag_present",
    "v43_job_base_count_bounded_flag",
    "v44_matrix_like_count_bounded_flag",
    "v45_max_parallel_jobs_bounded_flag",
    "v46_distinct_invocation_name_proxy_bounded_flag",
    "v47_invocation_candidate_partition_flag",
    "v48_execution_window_distinct_job_bounded_flag",
    "v49_matrix_expanded_implies_repeated_flag",
    "v50_cross_job_window_flag_consistency",
    "v51_parallel_same_style_flag_consistency",
    "v52_selected_invocation_priority_source_validity",
    "v53_style_level_complexity_class_presence",
]

# ----------------------------
# Summaries
# ----------------------------
summary_all = summarize_flags(df, flag_cols)
summary_all.to_csv(OUT_DIR / "validation_step1_summary.csv", index=False)

summary_by_style = summarize_flags(df, flag_cols, group_cols=["style"])
summary_by_style.to_csv(OUT_DIR / "validation_step1_style_summary.csv", index=False)

summary_by_controlled = summarize_flags(df, flag_cols, group_cols=["controlled_subset"])
summary_by_controlled.to_csv(OUT_DIR / "validation_step1_controlled_subset_summary.csv", index=False)

summary_by_style_controlled = summarize_flags(df, flag_cols, group_cols=["style", "controlled_subset"])
summary_by_style_controlled.to_csv(OUT_DIR / "validation_step1_style_controlled_subset_summary.csv", index=False)

# ----------------------------
# Record-level flags output
# ----------------------------
record_cols = key_cols + ["workflow_id", "workflow_identifier", "workflow_path", "run_attempt", "attempt", "style", "controlled_subset"]
record_cols = [c for c in record_cols if c in df.columns]
record_flags = df[record_cols + flag_cols].copy()
record_flags.to_csv(OUT_DIR / "validation_step1_record_flags.csv", index=False)

# ----------------------------
# Issue outputs
# ----------------------------
cutpoint_issue_rules = [
    "v5_cutpoint_temporal_order_flag",
    "v6_window_inside_run_flag",
    "v11_window_decomposition_flag",
    "v12_pre_invocation_recompute_flag",
    "v13_invocation_window_recompute_flag",
    "v14_post_invocation_recompute_flag",
    "v15_unique_invocation_cutpoint_flag",
    "v16_unique_execution_end_cutpoint_flag",
]
cutpoint_mismatches = df.loc[
    (df[cutpoint_issue_rules] == "mismatch").any(axis=1),
    record_cols + [
        "study_run_boundary_start_at",
        "study_run_boundary_end_at",
        "study_matched_invocation_step_started_at",
        "study_invocation_execution_end_step_completed_at",
        "study_invocation_execution_window_started_at",
        "study_invocation_execution_window_ended_at",
        "study_run_duration_seconds",
        "study_pre_invocation_selected_stage3_seconds",
        "study_invocation_execution_window_selected_stage3_seconds",
        "study_post_invocation_selected_stage3_seconds",
    ] + cutpoint_issue_rules
].copy()
cutpoint_mismatches.to_csv(OUT_DIR / "validation_step1_cutpoint_mismatches.csv", index=False)

step_match_rules = [
    "v15_unique_invocation_cutpoint_flag",
    "v16_unique_execution_end_cutpoint_flag",
    "v17_invocation_step_source_match_flag",
    "v18_execution_end_step_source_match_flag",
]
step_match_issues = df.loc[
    (df[step_match_rules] == "mismatch").any(axis=1),
    record_cols + [
        "inv_selected_count",
        "exe_selected_count",
        "study_matched_invocation_step_name",
        "study_matched_invocation_job_name",
        "study_matched_invocation_step_started_at",
        "study_matched_invocation_step_completed_at",
        "study_invocation_execution_end_step_name",
        "study_invocation_execution_end_job_name",
        "study_invocation_execution_end_step_started_at",
        "study_invocation_execution_end_step_completed_at",
        "inv_step_name_source",
        "inv_job_name_source",
        "inv_started_at_source",
        "inv_completed_at_source",
        "exe_step_name_source",
        "exe_job_name_source",
        "exe_started_at_source",
        "exe_completed_at_source",
    ] + step_match_rules
].copy()
step_match_issues.to_csv(OUT_DIR / "validation_step1_step_match_issues.csv", index=False)

aux_rules = [
    "v20_base_flag_present",
    "v21_robust_flag_present",
    "v22_style_distinct_job_count_present",
    "v23_style_distinct_job_base_count_present",
    "v24_style_matrix_like_job_count_present",
    "v25_style_matrix_expansion_flag_present",
    "v26_style_parallel_same_style_flag_present",
    "v27_style_max_parallel_jobs_present",
    "v28_style_repeated_same_style_flag_present",
    "v29_style_invocation_candidate_count_present",
    "v30_style_distinct_invocation_step_name_count_present",
    "v31_style_invocation_step_name_list_present",
    "v32_style_complexity_class_present",
    "v33_invocation_candidate_total_count_present",
    "v34_stage1_anchor_candidate_count_present",
    "v35_explicit_instru_candidate_count_present",
    "v36_custom_supported_candidate_count_present",
    "v37_distinct_invocation_candidate_step_name_count_present",
    "v38_distinct_invocation_candidate_job_count_present",
    "v39_selected_invocation_priority_source_present",
    "v40_execution_window_candidate_count_present",
    "v41_execution_window_distinct_job_count_present",
    "v42_cross_job_execution_window_flag_present",
    "v43_job_base_count_bounded_flag",
    "v44_matrix_like_count_bounded_flag",
    "v45_max_parallel_jobs_bounded_flag",
    "v46_distinct_invocation_name_proxy_bounded_flag",
    "v47_invocation_candidate_partition_flag",
    "v48_execution_window_distinct_job_bounded_flag",
    "v49_matrix_expanded_implies_repeated_flag",
    "v50_cross_job_window_flag_consistency",
    "v51_parallel_same_style_flag_consistency",
    "v52_selected_invocation_priority_source_validity",
    "v53_style_level_complexity_class_presence",
]
auxiliary_issues = df.loc[
    (df[aux_rules] == "mismatch").any(axis=1) | (df[aux_rules] == "missing").any(axis=1),
    record_cols + [
        "study_style_invocation_candidate_step_count_proxy",
        "study_style_invocation_candidate_step_names_proxy",
        "study_selected_invocation_priority_source",
        "study_stage1_anchor_candidate_count",
        "study_explicit_instru_candidate_count",
        "study_custom_supported_candidate_count",
        "study_invocation_candidate_count_total",
        "study_matched_invocation_step_started_at",
        "study_style_same_style_complexity_class",
    ] + aux_rules
].copy()
auxiliary_issues.to_csv(OUT_DIR / "validation_step1_auxiliary_issues.csv", index=False)

# All mismatches in controlled subset only
all_mismatch_mask = (df["controlled_subset"] == "Yes") & ((df[flag_cols] == "mismatch").any(axis=1))
all_mismatches_controlled = df.loc[all_mismatch_mask, record_cols + flag_cols].copy()
all_mismatches_controlled.to_csv(
    OUT_DIR / "validation_step1_all_mismatches_controlled_subset_only.csv",
    index=False
)

# ----------------------------
# Compact category-level view printed to screen
# ----------------------------
category_map = {
    "Structural integrity": [
        "v1_key_uniqueness_flag",
        "v2_required_ids_flag",
        "v19_style_scope_valid_flag",
        "v20_base_flag_present",
        "v21_robust_flag_present",
    ],
    "Temporal consistency": [
        "v3_run_bounds_order_flag",
        "v4_window_bounds_order_flag",
        "v5_cutpoint_temporal_order_flag",
        "v6_window_inside_run_flag",
    ],
    "Layer 1 equation validity": [
        "v10_layer1_sum_to_run_flag",
    ],
    "Layer 2 recomputation validity": [
        "v8_layer2_nonnegative_flag",
        "v9_layer2_bounded_by_run_flag",
        "v11_window_decomposition_flag",
        "v12_pre_invocation_recompute_flag",
        "v13_invocation_window_recompute_flag",
        "v14_post_invocation_recompute_flag",
    ],
    "Source traceability": [
        "v15_unique_invocation_cutpoint_flag",
        "v16_unique_execution_end_cutpoint_flag",
        "v17_invocation_step_source_match_flag",
        "v18_execution_end_step_source_match_flag",
    ],
    "Auxiliary consistency": [
        "v22_style_distinct_job_count_present",
        "v23_style_distinct_job_base_count_present",
        "v24_style_matrix_like_job_count_present",
        "v25_style_matrix_expansion_flag_present",
        "v26_style_parallel_same_style_flag_present",
        "v27_style_max_parallel_jobs_present",
        "v28_style_repeated_same_style_flag_present",
        "v29_style_invocation_candidate_count_present",
        "v30_style_distinct_invocation_step_name_count_present",
        "v31_style_invocation_step_name_list_present",
        "v32_style_complexity_class_present",
        "v33_invocation_candidate_total_count_present",
        "v34_stage1_anchor_candidate_count_present",
        "v35_explicit_instru_candidate_count_present",
        "v36_custom_supported_candidate_count_present",
        "v37_distinct_invocation_candidate_step_name_count_present",
        "v38_distinct_invocation_candidate_job_count_present",
        "v39_selected_invocation_priority_source_present",
        "v40_execution_window_candidate_count_present",
        "v41_execution_window_distinct_job_count_present",
        "v42_cross_job_execution_window_flag_present",
        "v43_job_base_count_bounded_flag",
        "v44_matrix_like_count_bounded_flag",
        "v45_max_parallel_jobs_bounded_flag",
        "v46_distinct_invocation_name_proxy_bounded_flag",
        "v47_invocation_candidate_partition_flag",
        "v48_execution_window_distinct_job_bounded_flag",
        "v49_matrix_expanded_implies_repeated_flag",
        "v50_cross_job_window_flag_consistency",
        "v51_parallel_same_style_flag_consistency",
        "v52_selected_invocation_priority_source_validity",
        "v53_style_level_complexity_class_presence",
    ],
}

def applicable_accuracy_for_rule(series):
    ok = int((series == "ok").sum())
    mm = int((series == "mismatch").sum())
    denom = ok + mm
    return np.nan if denom == 0 else (ok / denom) * 100.0

def category_summary(df_in):
    rows = []
    for cat, rules in category_map.items():
        vals = []
        for r in rules:
            acc = applicable_accuracy_for_rule(df_in[r])
            if not np.isnan(acc):
                vals.append(acc)
        rows.append({
            "category": cat,
            "avg_applicable_record_accuracy_pct": round(float(np.mean(vals)), 2) if vals else np.nan
        })
    return pd.DataFrame(rows)

print("\n=== Category summary: full dataset ===")
print(category_summary(df).to_string(index=False))

print("\n=== Category summary: controlled subset only ===")
print(category_summary(df[df["controlled_subset"] == "Yes"]).to_string(index=False))

print("\nSaved files to:", OUT_DIR)

C:\Users\gilla\AppData\Local\Temp\ipykernel_9264\999974204.py:193: DtypeWarning: Columns (25,26,39,41,49) have mixed types. Specify dtype option on import or set low_memory=False.
  steps = pd.read_csv(IN_STEPS_CSV)



=== Category summary: full dataset ===
                      category  avg_applicable_record_accuracy_pct
          Structural integrity                              100.00
          Temporal consistency                               99.55
     Layer 1 equation validity                              100.00
Layer 2 recomputation validity                              100.00
           Source traceability                               99.90
         Auxiliary consistency                              100.00

=== Category summary: controlled subset only ===
                      category  avg_applicable_record_accuracy_pct
          Structural integrity                               100.0
          Temporal consistency                               100.0
     Layer 1 equation validity                               100.0
Layer 2 recomputation validity                               100.0
           Source traceability                                99.9
         Auxiliary consistency         